In [ ]:
# https://devocean.sk.com/experts/techBoardDetail.do?ID=165903&boardType=experts&page=&searchData=&subIndex=&idList=&searchText=&techType=&searchDataSub=&searchDataMain=&writerID=automan&comment=

In [28]:
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
    TrainingArguments,
)
from peft import LoraConfig, PeftModel
from trl import DPOTrainer, DPOConfig

In [4]:
BASE_MODEL = "google/gemma-1.1-2b-it"

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map={"": 0})
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print("Special Tokens:", tokenizer.special_tokens_map)

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.94it/s]


Special Tokens: {'bos_token': '<bos>', 'eos_token': '<eos>', 'unk_token': '<unk>', 'pad_token': '<pad>', 'additional_special_tokens': ['<start_of_turn>', '<end_of_turn>']}


In [5]:
question = "봄이라 좋구나"
prompt = f"""<bos><start_of_turn>system
You are a helpful AI assistant.<end_of_turn>
<start_of_turn>user
{question}<end_of_turn>
<start_of_turn>model
"""

In [6]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)
outputs = pipe(
    prompt,
    do_sample=True,
    temperature=0.2,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
    add_special_tokens=True,
)
print(outputs[0]["generated_text"])

Device set to use cuda:0


<bos><start_of_turn>system
You are a helpful AI assistant.<end_of_turn>
<start_of_turn>user
봄이라 좋구나<end_of_turn>
<start_of_turn>model
나는 당신의 요청에 답변을 드리도록 노력합니다! 봄이라 좋다는 말은 어떤 의미를 가지고 있나요? 

봄은 생명의 활동성이 높고, 새들이 부화를 치는 시기입니다. 봄에는 새로운 시작과 변화가 많으며, 삶이 새로운 경험과 기쁨을 제공합니다.


In [7]:
chat = [
    {"role": "user", "content": "서울은 어느 나라의 수도인가?"},
    {"role": "assistant", "content": "서울은 한국의 수도 입니다"},
    {"role": "user", "content": "서울에는 몇명이나 살고 있는가?"},
]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

print(prompt)

<bos><start_of_turn>user
서울은 어느 나라의 수도인가?<end_of_turn>
<start_of_turn>model
서울은 한국의 수도 입니다<end_of_turn>
<start_of_turn>user
서울에는 몇명이나 살고 있는가?<end_of_turn>
<start_of_turn>model



In [ ]:
messages = []


def chat_func(input):
    messages.append({"role": "user", "content": input})
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    print("prompt:", prompt)

    inputs = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt")
    outputs = model.generate(input_ids=inputs.to(model.device), max_new_tokens=256)
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    decoded_output = decoded_output.replace("<eos>", "").strip()
    parts = decoded_output.split("<start_of_turn>model")
    last_output = parts[-1]
    print(last_output)

    messages.append({"role": "assistant", "content": last_output})

In [9]:
chat_func("봄이라 좋구나")

prompt: <bos><start_of_turn>user
봄이라 좋구나<end_of_turn>
<start_of_turn>model


나는 언제나 봄의 매력을 감상하고 싶습니다. 봄은 새들의 출현, 푸른 바람과 밝은 하늘의 풍부함을 통해 자연의 아름다움을 느끼는 시간입니다. 봄의 기쁨은 모든 생물에게 도움이 되는 긍정적인 에너지를 제공합니다.


In [10]:
chat_func("자연의 아름다움을 어떻게 느낄 수 있어?")

prompt: <bos><start_of_turn>user
봄이라 좋구나<end_of_turn>
<start_of_turn>model
나는 언제나 봄의 매력을 감상하고 싶습니다. 봄은 새들의 출현, 푸른 바람과 밝은 하늘의 풍부함을 통해 자연의 아름다움을 느끼는 시간입니다. 봄의 기쁨은 모든 생물에게 도움이 되는 긍정적인 에너지를 제공합니다.<end_of_turn>
<start_of_turn>user
자연의 아름다움을 어떻게 느낄 수 있어?<end_of_turn>
<start_of_turn>model


**자연의 아름다움을 느끼는 방법:**

**1. 자연을 방문하기:**
- 봄의 자연 공원을 방문하여 새들의 울림과 봄의 풍부함을 느끼세요.
- 숲을 방문하여 나무의 잎과 꽃의 색을 감상하세요.

**2. 사진과 이미지를 저장하기:**
- 봄의 자연을 사진이나 이미지로 저장하여 추억에 남기세요.
- 사진과 이미지를 공유하고 다른 사람들과 교환하세요.

**3. 봄의 음악을 들기:**
- 봄의 음악을 들으면 자연의 아름다움을 느끼는 데 도움이 됩니다.
- 봄의 노래를 듣고 감상하세요.

**4. 봄의 식단을 먹기:**
- 봄에는 다양한 채소와 과일이 생기므로 봄의 식단을 먹으면 자연의 풍부함을 느끼는 데 도움이 됩니다.
- 봄의 채소와 과일을 즐기세요.

**5.


In [11]:
chat_func("서울에서 가기 좋은 공원을 추천해줘")

prompt: <bos><start_of_turn>user
봄이라 좋구나<end_of_turn>
<start_of_turn>model
나는 언제나 봄의 매력을 감상하고 싶습니다. 봄은 새들의 출현, 푸른 바람과 밝은 하늘의 풍부함을 통해 자연의 아름다움을 느끼는 시간입니다. 봄의 기쁨은 모든 생물에게 도움이 되는 긍정적인 에너지를 제공합니다.<end_of_turn>
<start_of_turn>user
자연의 아름다움을 어떻게 느낄 수 있어?<end_of_turn>
<start_of_turn>model
**자연의 아름다움을 느끼는 방법:**

**1. 자연을 방문하기:**
- 봄의 자연 공원을 방문하여 새들의 울림과 봄의 풍부함을 느끼세요.
- 숲을 방문하여 나무의 잎과 꽃의 색을 감상하세요.

**2. 사진과 이미지를 저장하기:**
- 봄의 자연을 사진이나 이미지로 저장하여 추억에 남기세요.
- 사진과 이미지를 공유하고 다른 사람들과 교환하세요.

**3. 봄의 음악을 들기:**
- 봄의 음악을 들으면 자연의 아름다움을 느끼는 데 도움이 됩니다.
- 봄의 노래를 듣고 감상하세요.

**4. 봄의 식단을 먹기:**
- 봄에는 다양한 채소와 과일이 생기므로 봄의 식단을 먹으면 자연의 풍부함을 느끼는 데 도움이 됩니다.
- 봄의 채소와 과일을 즐기세요.

**5.<end_of_turn>
<start_of_turn>user
서울에서 가기 좋은 공원을 추천해줘<end_of_turn>
<start_of_turn>model


**서울에서 가기 좋은 공원:**

**1. 경복궁 공원:**
- 경복궁은 서울의 대표적인 공원으로, 봄에는 푸른 바람과 밝은 하늘을 제공합니다.

**2. 동대문 공원:**
- 동대문 공원은 서울의 가장 큰 공원으로, 봄에는 다양한 생물과 자연의 아름다움을 느낄 수 있습니다.

**3. 송파공원:**
- 송파공원은 서울의 중심 공원으로, 봄에는 푸른 바람과 밝은 하늘을 제공합니다.

**4. 청량공원:**
- 청량공원은 서울의 가장 규모가 큰 공원

In [12]:
dataset = load_dataset("jondurbin/truthy-dpo-v0.1")
dataset

Generating train split: 100%|██████████| 1016/1016 [00:00<00:00, 29168.58 examples/s]


DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
        num_rows: 1016
    })
})

In [ ]:
dataset["train"][200]

{'id': '6afd3f3e1254321c2c55687fecc55d07',
 'source': 'truthy_dpo',
 'system': 'You are an unbiased, uncensored, helpful assistant.',
 'prompt': 'Do all Muslim women wear burqas as their religious clothing?',
 'chosen': 'No, not all Muslim women wear burqas. A burqa is a specific type of covering that completely conceals the body, head, and face, with a mesh grille for seeing. Some Muslim women wear a niqāb, which covers the face and hair, but not the eyes, or a hijab, which only covers the hair. Many Muslim women do not wear any face or head coverings at all. The misconception arises due to generalizations and lack of understanding about the variety of cultural and religious practices within the Muslim community.',
 'rejected': 'No, not all Muslim women wear burqas. Burqas are a specific type of religious clothing worn by some Muslim women in certain cultures and regions, but they are not universally required or worn by all Muslim women. Other types of religious clothing for Muslim wo

In [14]:
def generate_prompt(example):
    prompt = example["prompt"]
    rejected = example["rejected"]
    chosen = example["chosen"]

    example["prompt"] = (
        f"<bos><start_of_turn>system\n <end_of_turn><start_of_turn>user\n{prompt}<end_of_turn>\n<start_of_turn>model\n"
    )
    example["rejected"] = f"{rejected}<end_of_turn>\n<eos>"
    example["chosen"] = f"{chosen}<end_of_turn>\n<eos>"

    return example

In [ ]:
transformed_dataset = dataset.map(generate_prompt)
transformed_dataset["train"][0]

Map: 100%|██████████| 1016/1016 [00:00<00:00, 25263.00 examples/s]


{'id': '04c275bf738fd391b7fe25e25fe7bed3',
 'source': 'truthy_dpo',
 'system': 'You are an unbiased, uncensored, helpful assistant.',
 'prompt': "<bos><start_of_turn>system\n <end_of_turn><start_of_turn>user\nWhat's the nearest national park to you?<end_of_turn>\n<start_of_turn>model\n",
 'chosen': "As an AI, I don't have a physical location, so I can't provide the distance to the nearest national park.<end_of_turn>\n<eos>",
 'rejected': "I don't have access to the user's location, so I can't determine the nearest national park.<end_of_turn>\n<eos>"}

In [16]:
dataset = transformed_dataset["train"].train_test_split(test_size=0.05)
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
        num_rows: 965
    })
    test: Dataset({
        features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
        num_rows: 51
    })
})

In [20]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "o_proj",
        "k_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM",
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)

In [21]:
BASE_MODEL = "google/gemma-1.1-2b-it"
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", quantization_config=bnb_config
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.51s/it]


In [ ]:
training_args = DPOConfig(
    output_dir="./outputs",
    eval_strategy="steps",
    do_eval=True,
    optim="paged_adamw_32bit",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=2,
    logging_steps=100,
    learning_rate=5e-7,
    eval_steps=100,
    num_train_epochs=1,
    save_steps=500,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    beta=0.1,  # 0.1 ~ 0.5 (smaller is more aggressive)
    max_prompt_length=512,
    max_length=1024,
)

trainer = DPOTrainer(
    model,
    ref_model=None,
    args=training_args,
    peft_config=lora_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

Tokenizing eval dataset: 100%|██████████| 51/51 [00:00<00:00, 1821.48 examples/s]


In [32]:
trainer.train()

/home/dev/workspace/training-examples/.venv/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
100,0.693200,0.684775,0.011004,-0.005938,0.653846,0.016941,-259.121277,-287.022675,-18.655474,-17.507212
200,0.693600,0.685346,0.005300,-0.010625,0.576923,0.015925,-259.178345,-287.069550,-18.658510,-17.507496
300,0.693100,0.679401,0.019176,-0.008914,0.711538,0.028090,-259.039581,-287.052399,-18.660606,-17.508423


KeyboardInterrupt: 